# Lab 03 (จุดเริ่มต้น) · Agent ที่เรียกใช้ระบบ RAG
**1 ต.ค. 10.00–12.00** — notebook นี้เป็นโครงเริ่มต้น ต่อยอดเป็น docs agent / frontend ได้เลย

```
ผู้ใช้ ──▶ Agent (LangChain + LLM บน vLLM)
              │  ตัดสินใจเองว่าจะค้นอะไร กี่ครั้ง
              ▼
        search_documents  (tool สำเร็จรูป: src/cdd_rag/tools.py)
              │
              ▼
        ระบบ RAG จาก Lab 01  (hybrid search บน Qdrant — ข้อมูลที่ index ไว้แล้ว)
```

**สิ่งที่ต้องมี:** `.env` ชุดเดียวกับ Lab 01 (ค่า `GROUP` เดียวกัน → collection เดียวกัน) + ตั้ง `LLM_MODEL` · ไม่ต้อง index ใหม่

In [ ]:
from cdd_rag.config import settings
from cdd_rag.store import client

print("collection :", settings.collection, "| points:", client.count(settings.collection).count)
print("LLM        :", settings.llm_base_url, settings.llm_model or "⚠️ ยังไม่ได้ตั้ง LLM_MODEL ใน .env")

## 1 · Tool: สิ่งที่ agent "เห็น" เมื่อค้นเอกสาร
ผลแต่ละชิ้นมี **รหัสอ้างอิง** เช่น `[da7f4c_c004]` — ไม่ใช้ `[1]` `[2]` เพราะ agent อาจค้นหลายครั้งในคำถามเดียว เลขจะชนกัน

In [ ]:
from cdd_rag.tools import run_search_documents

text, hits = run_search_documents("สอนเรื่อง Docker วันไหน")
print(text[:500])
print("\nข้อมูลสำหรับ frontend (hit แรก):")
print({k: hits[0][k] for k in ("ref", "file_name", "page", "section", "link")})

## 2 · สร้าง agent
- `search_documents` = tool สำเร็จรูป (LangChain) · `make_llm()` = LLM บน vLLM ตาม `.env`
- system prompt บอก 3 เรื่อง: ต้องค้นก่อนตอบ · ต้องอ้างอิงรหัส · ไม่พบให้บอกว่าไม่พบ

In [ ]:
from langchain.agents import create_agent
from cdd_rag.tools import search_documents, make_llm, collect_sources, link_citations

SYSTEM_PROMPT = """คุณเป็นผู้ช่วยตอบคำถามจากเอกสารของกรมการพัฒนาชุมชน
- ต้องใช้เครื่องมือ search_documents ค้นเอกสารก่อนตอบทุกครั้ง ค้นได้หลายครั้งถ้าคำถามมีหลายเรื่อง
- ตอบจากผลการค้นเท่านั้น และใส่รหัสอ้างอิงในวงเล็บเหลี่ยมท้ายประโยค เช่น [da7f4c_c004]
- ถ้าค้นแล้วไม่พบข้อมูล ให้ตอบว่า "ไม่พบข้อมูลในเอกสาร" ห้ามเดา
- ตอบเป็นภาษาไทย กระชับ"""

agent = create_agent(make_llm(), tools=[search_documents], system_prompt=SYSTEM_PROMPT) if settings.llm_model else None
print("agent พร้อม" if agent else "ข้าม: ยังไม่ได้ตั้ง LLM_MODEL")

## 3 · ถามคำถาม แล้วดูว่า agent ทำอะไรบ้าง

In [ ]:
def ask(question):
    if agent is None:
        print("ข้าม: ยังไม่ได้ตั้ง LLM_MODEL")
        return
    result = agent.invoke({"messages": [("user", question)]})
    messages = result["messages"]

    print("❓", question)
    for m in messages:                                   # agent เรียก tool อะไร ด้วยคำค้นอะไร
        for call in getattr(m, "tool_calls", None) or []:
            print("   🔧", call["name"], call["args"])

    sources = collect_sources(messages)                  # รวมผลจากทุกครั้งที่ค้น
    answer = messages[-1].content
    print("\n💬", answer)
    print("\n📎 อ้างอิง (สำหรับทำปุ่มเปิด PDF):")
    for ref, h in sources.items():
        if f"[{ref}]" in answer:
            print(f"   [{ref}] {h['file_name']} หน้า {h['page']} → {h['link']}")
    return link_citations(answer, sources)               # คำตอบแบบ Markdown ที่รหัสกลายเป็นลิงก์

ask("การอบรมเรื่อง Docker จัดวันไหน และใครเป็นวิทยากร")

In [ ]:
# คำถามที่ต้องค้น 2 เรื่อง → agent ควรเรียก tool 2 ครั้ง
ask("หัวข้อ Docker สอนวันไหน และการอบรมจัดที่ห้องอะไร")

In [ ]:
# คำถามที่ไม่มีคำตอบในเอกสาร → ควรตอบว่าไม่พบ ไม่ใช่เดา
ask("ค่าเบี้ยเลี้ยงวิทยากรต่อวันเท่าไหร่")

## ต่อยอด
| อยากทำ | ใช้ |
|---|---|
| ปุ่มเปิด PDF ไปหน้าที่อ้างอิง | `collect_sources()` → `hit["link"]` = `/files/{file_path}#page={page}` · `link_citations()` แปลงรหัสในคำตอบเป็นลิงก์ |
| ให้ agent ค้นเฉพาะประเภทเอกสาร | tool รับ `doc_type`, `file_name` อยู่แล้ว — agent เลือกเองจากคำถาม |
| ไม่ใช้ LangChain | `from cdd_rag.tools import OPENAI_TOOL, run_search_documents` (ดู `docs/HANDOFF.md`) |

⚠️ **ห้าม** ingest เอกสารผ่าน LangChain / LlamaIndex เอง และห้ามเปลี่ยนชื่อ field ใน Qdrant — ดู `docs/data-contract.md`